In [1]:
import festim as F
import ufl
import numpy as np

print(F.__version__)

/Users/ckhurana/miniconda3/envs/festim-dev/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2.2rc2.dev62+gd40ae63c7


In [2]:
my_model = F.HydrogenTransportProblem()
my_model.mesh    = F.Mesh1D(vertices=np.linspace(0, 1e-3, 1001))  # 1 mm domain
material = F.Material(D_0=1.08e-7, E_D=0.46)

volume = F.VolumeSubdomain1D(id=1, borders=[0, 1e-3], material=material)
left = F.SurfaceSubdomain1D(id=1, x=0)
right = F.SurfaceSubdomain1D(id=2, x=1e-3)

my_model.subdomains = [volume, left, right]

In [3]:

# =============================================================================
# Species
# =============================================================================

C_SS   = F.Species("C_SS")                    # mobile — diffusion + Soret
C_prec = F.Species("C_prec", mobile=False)    # immobile — reaction only

# =============================================================================
# Material parameters (Table 1, Passelaigue et al. 2021)
# =============================================================================

R_eV   = 8.617333e-5   # eV/K

TSS_D0 = 1.02e5;   Q_D  = 0.37    # wt.ppm, eV
TSS_P0 = 3.08e4;   Q_P  = 0.26
KD0    = 1.11e3;   E_D  = 0.46
KN0    = 2.75e-5
Eth0, Eth1, Eth2, Eth3 = 5.66e-1, 4e-4, 2e-7, 3e-10
KG0    = 1.71e-4   # override — Table 1 incomplete
p_jmak = 2.5


In [4]:
eps_prec  = 1e-6    # seed: prevents JMAK locking at C_prec = 0
eps_upper = 1e-14   # guards ln(1-x) from singularity

def dissolution_rate(T, c_SS, **kwargs):
    TSS_D = TSS_D0 * ufl.exp(-Q_D / (R_eV * T))
    K_D   = KD0   * ufl.exp(-E_D  / (R_eV * T))
    return ufl.conditional(
        ufl.lt(c_SS, TSS_D),
        K_D * (TSS_D - c_SS),
        ufl.zero()
    )

def nucleation_rate(T, c_SS, **kwargs):
    Eth   = -Eth0 + Eth1*T - Eth2*T**2 + Eth3*T**3
    TSS_P = TSS_P0 * ufl.exp(-Q_P / (R_eV * T))
    K_N   = KN0   * ufl.exp(-Eth  / (R_eV * T))
    return ufl.conditional(
        ufl.gt(c_SS, TSS_P),
        K_N * (c_SS - TSS_P),
        ufl.zero()
    )

def growth_rate(T, c_SS, c_prec, **kwargs):
    TSS_D       = TSS_D0 * ufl.exp(-Q_D / (R_eV * T))
    C_tot       = c_SS + c_prec
    denom       = C_tot - TSS_D
    x           = (c_prec + eps_prec) / denom
    one_minus_x = ufl.max_value(eps_upper, 1.0 - x)
    jmak        = p_jmak * one_minus_x * (-ufl.ln(one_minus_x))**(1.0 - 1.0/p_jmak)
    return ufl.conditional(
        ufl.And(ufl.gt(c_SS, TSS_D), ufl.gt(c_prec, ufl.zero())),
        KG0 * denom * jmak,
        ufl.zero()
    )

In [5]:

dissolution = F.ReactionBase(
    reaction_rate  = dissolution_rate,
    volume         = volume,
    reactant       = C_prec,    # C_prec is consumed
    product        = C_SS,      # C_SS is produced
    arg_to_species = {"c_SS": C_SS},
)

nucleation = F.ReactionBase(
    reaction_rate  = nucleation_rate,
    volume         = volume,
    reactant       = C_SS,      # C_SS is consumed
    product        = C_prec,    # C_prec is produced
    arg_to_species = {"c_SS": C_SS},
)

growth = F.ReactionBase(
    reaction_rate  = growth_rate,
    volume         = volume,
    reactant       = C_SS,      # C_SS is consumed
    product        = C_prec,    # C_prec is produced
    arg_to_species = {"c_SS": C_SS, "c_prec": C_prec},
)


In [6]:


my_model.species = [C_SS, C_prec]

my_model.temperature = 650   # K uniform, or fem.Function from HeatProblem

my_model.reactions = [dissolution, nucleation, growth]

my_model.initial_conditions = [
    F.InitialConcentration(species=C_SS,   value=288.0, volume=volume),   # wt.ppm
    F.InitialConcentration(species=C_prec, value=1e-6, volume=volume),    # JMAK seed
]

my_model.settings = F.Settings(atol=1e-10, rtol=1e-10, final_time=8000)
my_model.settings.stepsize = F.Stepsize(50.0)

my_model.exports = [
    F.VTXSpeciesExport(field=C_SS,   filename="C_SS.bp"),
    F.VTXSpeciesExport(field=C_prec, filename="C_prec.bp"),
]

my_model.initialise()
my_model.run()

ld: warning: duplicate -rpath '/Users/ckhurana/miniconda3/envs/festim-dev/lib' ignored
ld: warning: duplicate -rpath '/Users/ckhurana/miniconda3/envs/festim-dev/lib' ignored
ld: warning: duplicate -rpath '/Users/ckhurana/miniconda3/envs/festim-dev/lib' ignored
Solving HydrogenTransportProblem: 100%|██████████| 8.00k/8.00k [00:05<00:00, 1.54kit/s]
